# 01 — Data audit

Predict construction cycle time (days) using only what could be known **on the day construction starts**.

**Target (fixed):** `construction_end_date - construction_start_date` in calendar days. Inspection dates are used only as a data-quality check, never as the target.

This notebook inspects `data/build_history.csv` and `data/homes_to_predict.csv`. It does **not** modify those files, and it does not clean, impute, engineer features, split data, or train models. Diagnostic columns exist only in memory.

Run from the project root, top to bottom.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA = Path("data")
HIST_PATH, PRED_PATH = DATA / "build_history.csv", DATA / "homes_to_predict.csv"
assert HIST_PATH.exists() and PRED_PATH.exists(), "Run from the project root."

hist = pd.read_csv(HIST_PATH)
pred = pd.read_csv(PRED_PATH)
hist_lit = pd.read_csv(HIST_PATH, keep_default_na=False, na_values=[""])
pred_lit = pd.read_csv(PRED_PATH, keep_default_na=False, na_values=[""])

DATE_COLS = [
    "sale_date", "permit_application_date", "permit_issue_date",
    "construction_start_date", "construction_end_date", "final_inspection_date",
]
hist_dt, pred_dt = hist.copy(), pred.copy()
for c in DATE_COLS:
    hist_dt[c] = pd.to_datetime(hist[c], errors="coerce")
    pred_dt[c] = pd.to_datetime(pred[c], errors="coerce")

print(f"history {hist.shape}  |  predict {pred.shape}  |  pandas {pd.__version__}")


history (4684, 30)  |  predict (578, 30)  |  pandas 2.3.3


## 1. Schema, types, and missingness

Same 30 columns, same order. Predict dtypes differ only where a column is entirely empty (`float64` instead of date/int). `beds` is float in both because of missing values. `garage_size` is object because it mixes `1/2/3` with words.


In [2]:
cols_match = list(hist.columns) == list(pred.columns)
schema = pd.DataFrame({
    "dtype_hist": hist.dtypes.astype(str),
    "dtype_pred": pred.dtypes.astype(str),
    "miss_hist": hist.isna().sum(),
    "pct_hist": (hist.isna().mean() * 100).round(2),
    "miss_pred": pred.isna().sum(),
    "pct_pred": (pred.isna().mean() * 100).round(2),
})
schema["dtype_ok"] = schema["dtype_hist"] == schema["dtype_pred"]
schema.index.name = "column"

print("columns identical and in the same order:", cols_match)
print("dtype mismatches:", int((~schema["dtype_ok"]).sum()),
      "→", list(schema.index[~schema["dtype_ok"]]))
display(schema)

print("pandas default NA tokens turn basement_type 'None' into missing. Literal tokens:")
display(pd.DataFrame({
    "hist_literal": hist_lit["basement_type"].value_counts(dropna=False),
    "pred_literal": pred_lit["basement_type"].value_counts(dropna=False),
}).fillna(0).astype(int))


columns identical and in the same order: True
dtype mismatches: 3 → ['construction_end_date', 'final_inspection_date', 'change_order_count']


,dtype_hist,dtype_pred,miss_hist,pct_hist,miss_pred,pct_pred,dtype_ok
column,,,,,,,
home_id,object,object,0,0.00,0,0.00,True
community,object,object,0,0.00,0,0.00,True
metro,object,object,0,0.00,0,0.00,True
permit_authority,object,object,0,0.00,0,0.00,True
lot_number,int64,int64,0,0.00,0,0.00,True
plan_name,object,object,0,0.00,0,0.00,True
plan_code,object,object,0,0.00,0,0.00,True
product_line,object,object,0,0.00,0,0.00,True
beds,float64,float64,141,3.01,17,2.94,True


pandas default NA tokens turn basement_type 'None' into missing. Literal tokens:


,hist_literal,pred_literal
basement_type,,
Finished,978,103
None,1654,260
Unfinished,2052,215


## 2. Duplicates and identifiers

`home_id` (`H` + 6 digits) is the only unique key. Lot numbers reuse across communities. 25 history `home_id`s are exact full-row copies, not conflicting versions. No id overlap between files.


In [3]:
def id_report(df):
    dup_ids = df["home_id"].duplicated(keep=False)
    conflict = False
    if dup_ids.any():
        conflict = bool((df.loc[dup_ids].groupby("home_id").nunique().max(axis=1) > 1).any())
    return pd.Series({
        "rows": len(df),
        "unique_home_id": df["home_id"].nunique(),
        "exact_dup_extras": int(df.duplicated().sum()),
        "home_ids_copied": int(df.loc[dup_ids, "home_id"].nunique()),
        "conflicting_copies": conflict,
        "id_pattern_ok": bool(df["home_id"].astype(str).str.match(r"^H\d{6}$").all()),
    })

print("id overlap hist ∩ pred:", len(set(hist.home_id) & set(pred.home_id)))
print("hist ids", hist.home_id.min(), "→", hist.home_id.max(),
      "| pred ids", pred.home_id.min(), "→", pred.home_id.max())
display(pd.concat([id_report(hist).rename("history"), id_report(pred).rename("predict")], axis=1))

h = hist.assign(c=hist.community.str.strip().str.casefold())
print("lot_number unique:", hist.lot_number.nunique(),
      "| extra (community, lot) collisions:", int(h.duplicated(["c", "lot_number"]).sum()),
      "| extra (community, lot, plan) collisions:", int(h.duplicated(["c", "lot_number", "plan_code"]).sum()))


id overlap hist ∩ pred: 0
hist ids H100000 → H104658 | pred ids H104659 → H105236


,history,predict
rows,4684,578
unique_home_id,4659,578
exact_dup_extras,25,0
home_ids_copied,25,0
conflicting_copies,False,False
id_pattern_ok,True,True


lot_number unique: 419 | extra (community, lot) collisions: 1474 | extra (community, lot, plan) collisions: 137


## 3. Numeric validity

No negatives. Zeros are valid for `half_baths` and `change_order_count`. `beds` values are whole numbers 2–5.

The only physically implausible numeric defect is **`sqft` extra zeros**: 13 history + 3 predict rows above 10,000, each ~10× that plan’s median. IQR fence from history (~4,560) flags the same rows. House sqft > lot sqft can be valid on 2-storey homes; most of those 49 history cases are these outliers.


In [4]:
NUM = ["beds", "baths", "half_baths", "sqft", "stories", "lot_size_sqft",
       "selected_options_value", "change_order_count", "trade_invoice_total"]

print("history")
display(hist[NUM].describe(percentiles=[0.05, 0.5, 0.95]).T)
print("predict  (change_order_count / trade_invoice_total are 100% missing)")
display(pred[NUM].describe(percentiles=[0.05, 0.5, 0.95]).T)

flags = []
for name, df in [("history", hist), ("predict", pred)]:
    for col in NUM:
        s = df[col]
        flags.append({
            "dataset": name, "column": col,
            "neg": int((s < 0).sum()) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "zero": int((s == 0).sum()) if pd.api.types.is_numeric_dtype(s) else np.nan,
            "non_int": int(((s.dropna() % 1) != 0).sum()) if pd.api.types.is_numeric_dtype(s) else np.nan,
        })
display(pd.DataFrame(flags).pivot(index="column", columns="dataset", values=["neg", "zero", "non_int"]))

plan_med = hist.groupby("plan_name")["sqft"].median()
q1, q3 = hist.sqft.quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print(f"history IQR upper fence for sqft: {fence:.0f}")

def sqft_outliers(df, label):
    t = df.loc[df.sqft > 10000, ["home_id", "plan_name", "product_line", "sqft", "lot_size_sqft"]].copy()
    t["plan_median"] = t.plan_name.map(plan_med)
    t["ratio"] = (t.sqft / t.plan_median).round(2)
    t.insert(0, "dataset", label)
    return t.sort_values("sqft", ascending=False)

display(pd.concat([sqft_outliers(hist, "history"), sqft_outliers(pred, "predict")], ignore_index=True))

print("selected_options_value > 100,000  hist/pred:",
      int((hist.selected_options_value > 100000).sum()), "/",
      int((pred.selected_options_value > 100000).sum()),
      "| spec vs sold mean options (hist):")
display(hist.groupby("is_spec_home")["selected_options_value"].describe()[["count", "mean", "50%", "max"]])
print("trade_invoice_total by status (hist) — low totals exist only while in progress:")
display(hist.groupby("construction_status")["trade_invoice_total"].describe()[["count", "min", "mean", "50%", "max"]])


history


,count,mean,std,min,5%,50%,95%,max
column,,,,,,,,
beds,"4,543.00",3.53,0.96,2.00,2.00,3.00,5.00,5.00
baths,"4,684.00",3.05,0.91,2.00,2.00,3.00,5.00,5.00
half_baths,"4,684.00",0.32,0.47,0.00,0.00,0.00,1.00,1.00
sqft,"4,684.00","2,126.92","1,405.82","1,050.00","1,190.00","1,950.00","3,350.00","34,900.00"
stories,"4,684.00",1.88,0.32,1.00,1.00,2.00,2.00,2.00
lot_size_sqft,"4,587.00","4,362.20","1,769.18","1,500.00","2,050.00","4,150.00","7,350.00","10,200.00"
selected_options_value,"4,684.00","27,544.75","19,074.73",200.00,"3,600.00","24,400.00","62,785.00","145,000.00"
change_order_count,"4,684.00",1.67,1.56,0.00,0.00,1.00,5.00,8.00
trade_invoice_total,"4,684.00","488,685.18","182,893.87","2,400.00","177,575.00","476,550.00","793,685.00","1,306,100.00"


predict  (change_order_count / trade_invoice_total are 100% missing)


,count,mean,std,min,5%,50%,95%,max
beds,561.00,3.30,0.94,2.00,2.00,3.00,5.00,5.00
baths,578.00,2.84,0.85,2.00,2.00,3.00,4.00,5.00
half_baths,578.00,0.34,0.47,0.00,0.00,0.00,1.00,1.00
sqft,578.00,"1,973.20","1,615.46","1,050.00","1,170.00","1,680.00","3,231.50","27,700.00"
stories,578.00,1.90,0.30,1.00,1.00,2.00,2.00,2.00
lot_size_sqft,566.00,"3,913.52","1,718.18","1,500.00","2,000.00","3,425.00","7,000.00","9,050.00"
selected_options_value,578.00,"27,931.31","19,520.79",700.00,"3,600.00","24,900.00","66,060.00","104,500.00"
change_order_count,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trade_invoice_total,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


neg            zero         non_int        
dataset                history predict history predict history predict
column                                                                
baths                        0       0       0       0       0       0
beds                         0       0       0       0       0       0
change_order_count           0       0    1384       0       0       0
half_baths                   0       0    3190     382       0       0
lot_size_sqft                0       0       0       0       0       0
selected_options_value       0       0       0       0       0       0
sqft                         0       0       0       0       0       0
stories                      0       0       0       0       0       0
trade_invoice_total          0       0       0       0       0       0

...

,dataset,home_id,plan_name,product_line,sqft,lot_size_sqft,plan_median,ratio
0,history,H100569,Nyssa,Single Family,34900,NaN,"3,410.00",10.23
1,history,H102804,Larch,Single Family,30800,"7,000.00","2,860.00",10.77
2,history,H100489,Larch,Single Family,30600,"5,050.00","2,860.00",10.70
3,history,H100450,Maple,Single Family,30500,"3,500.00","3,090.00",9.87
4,history,H104295,Larch,Single Family,28400,"5,850.00","2,860.00",9.93
5,history,H104499,Larch,Single Family,27400,"5,050.00","2,860.00",9.58
6,history,H102544,Katsura,Single Family,26200,"6,750.00","2,610.00",10.04
7,history,H100767,Oakmont,Single Family,23100,"5,300.00","2,200.00",10.50
8,history,H101916,Oakmont,Single Family,22000,NaN,"2,200.00",10.00
9,history,H100969,Hawthorn,Single Family,19400,"6,750.00","1,840.00",10.54


selected_options_value > 100,000  hist/pred: 14 / 1 | spec vs sold mean options (hist):


,count,mean,50%,max
is_spec_home,,,,
False,"3,603.00","33,277.02","29,900.00","145,000.00"
True,"1,081.00","8,438.95","7,100.00","38,300.00"


trade_invoice_total by status (hist) — low totals exist only while in progress:


,count,min,mean,50%,max
construction_status,,,,,
Complete,"4,105.00","226,700.00","522,026.99","500,500.00","1,306,100.00"
In Progress,579.00,"2,400.00","252,298.10","222,100.00","858,000.00"


## 4. Dates, cycle-time target, and in-progress rows

All non-empty dates are ISO `YYYY-MM-DD` and parse. Permit issue is never before application; start is never before issue. Every recorded `sale_date` is on or before start, so a known sale is known at start.

**Target (fixed):** `cycle_days = construction_end_date - construction_start_date`. Defined for all 4,105 Complete rows; undefined for 579 In Progress history rows and all 578 scoring homes. Do not substitute `final_inspection_date`.

Seven Complete rows have end **before** start, so the target is negative and unusable. Those seven are also the only rows with inspection after end (inspection-minus-start is 108–313 days). That is evidence the stored *end* dates are wrong, not a license to change the target formula. On the other 4,098 completes, end is 1–4 days after inspection (median 3).

**Truncation:** history stops 2025-07-31; scoring starts 2025-08-01. Of 703 history starts in 2025, 547 are still in progress. The 156 that finished by the snapshot have a much shorter mean cycle (~130 vs ~187 days). Excluding in-progress rows is required for a fully observed target and length-biases recent cohorts.


In [5]:
def parse_check(raw, name):
    rows = []
    for col in DATE_COLS:
        s = raw[col].astype(str)
        nonempty = s[s.notna() & ~s.isin(["", "nan"])]
        parsed = pd.to_datetime(nonempty, errors="coerce")
        rows.append({
            "dataset": name, "column": col, "n_nonempty": len(nonempty),
            "unparseable": int(parsed.isna().sum()),
            "non_iso": int((~nonempty.str.match(r"^\d{4}-\d{2}-\d{2}$")).sum()),
            "min": parsed.min() if parsed.notna().any() else pd.NaT,
            "max": parsed.max() if parsed.notna().any() else pd.NaT,
        })
    return pd.DataFrame(rows)

display(pd.concat([parse_check(hist_lit, "history"), parse_check(pred_lit, "predict")], ignore_index=True))

def order_flags(df):
    s, a, i, st, e, f = [df[c] for c in DATE_COLS]
    checks = {
        "issue < application": (i < a),
        "start < issue": (st < i),
        "end < start": e.notna() & (e < st),
        "inspection < start": f.notna() & (f < st),
        "inspection > end": e.notna() & f.notna() & (f > e),
        "sale > start": s.notna() & (s > st),
        "sale > issue": s.notna() & (s > i),
    }
    return pd.Series({k: int(v.sum()) for k, v in checks.items()})

print("date-order flags (n rows)")
display(pd.DataFrame({"history": order_flags(hist_dt), "predict": order_flags(pred_dt)}))

print("status vs missing end date (history):")
display(pd.crosstab(hist.construction_status, hist.construction_end_date.isna(), colnames=["end_missing"]))

inv = hist_dt["construction_end_date"].notna() & (
    hist_dt["construction_end_date"] < hist_dt["construction_start_date"]
)
inv_tbl = hist_dt.loc[inv, ["home_id", "construction_start_date", "construction_end_date",
                            "final_inspection_date", "product_line"]].copy()
inv_tbl["end_minus_start"] = (inv_tbl.construction_end_date - inv_tbl.construction_start_date).dt.days
inv_tbl["insp_minus_start"] = (inv_tbl.final_inspection_date - inv_tbl.construction_start_date).dt.days
print("seven inverted ends (target = end - start is negative; inspection shown only as a quality check):")
display(inv_tbl.sort_values("home_id"))

complete = hist_dt[hist_dt.construction_status.eq("Complete")].copy()
complete["cycle_days"] = (complete.construction_end_date - complete.construction_start_date).dt.days
print("cycle_days = construction_end_date - construction_start_date (Complete rows only):")
display(complete["cycle_days"].describe(percentiles=[0.05, 0.5, 0.95]).to_frame("cycle_days"))
print("negative / zero cycle_days:", int((complete.cycle_days < 0).sum()), "/",
      int((complete.cycle_days == 0).sum()),
      "(negative rows cannot be used until the end date is corrected)")
display(complete.groupby("product_line")["cycle_days"].describe()[["count", "mean", "50%", "max"]])

complete["start_year"] = complete.construction_start_date.dt.year
print("cycle_days by start year (2025 completes are truncated):")
display(complete.groupby("start_year")["cycle_days"].describe()[["count", "mean", "50%", "max"]])

ip = hist_dt[hist_dt.construction_status.eq("In Progress")]
print("in-progress start min/max:", ip.construction_start_date.min().date(),
      ip.construction_start_date.max().date(),
      "| by year:", ip.construction_start_date.dt.year.value_counts().sort_index().to_dict())
print("snapshot: hist start/end ≤ 2025-07-31:",
      bool((hist_dt.construction_start_date <= "2025-07-31").all()),
      bool(((hist_dt.construction_end_date.isna()) | (hist_dt.construction_end_date <= "2025-07-31")).all()),
      "| pred start ≥ 2025-08-01:", bool((pred_dt.construction_start_date >= "2025-08-01").all()))


,dataset,column,n_nonempty,unparseable,non_iso,min,max
0,history,sale_date,3099,0,0,2020-08-01,2025-06-28
1,history,permit_application_date,4684,0,0,2020-07-18,2025-07-07
2,history,permit_issue_date,4684,0,0,2020-11-14,2025-07-22
3,history,construction_start_date,4684,0,0,2021-01-04,2025-07-31
4,history,construction_end_date,4105,0,0,2021-05-12,2025-07-31
5,history,final_inspection_date,4105,0,0,2021-05-10,2025-07-29
6,predict,sale_date,382,0,0,2025-03-05,2025-11-24
7,predict,permit_application_date,578,0,0,2025-01-18,2025-12-02
8,predict,permit_issue_date,578,0,0,2025-06-07,2025-12-23
9,predict,construction_start_date,578,0,0,2025-08-01,2025-12-31


date-order flags (n rows)


,history,predict
issue < application,0,0
start < issue,0,0
end < start,7,0
inspection < start,0,0
inspection > end,7,0
sale > start,0,0
sale > issue,241,28


status vs missing end date (history):


end_missing,False,True
construction_status,,
Complete,4105,0
In Progress,0,579


...

cycle_days by start year (2025 completes are truncated):


,count,mean,50%,max
start_year,,,,
2021,829.00,178.47,173.00,571.00
2022,984.00,224.61,220.00,622.00
2023,"1,045.00",190.93,182.00,449.00
2024,"1,091.00",164.40,157.00,362.00
2025,156.00,130.29,131.00,205.00


in-progress start min/max:

 2024-08-04 2025-07-31 | by year: {2024: 32, 2025: 547}


snapshot: hist start/end ≤ 2025-07-31: True True | pred start ≥ 2025-08-01: True


## 5. Categoricals, unseen labels, and train vs score mix

Only `community` is dirty (whitespace / case). After strip + casefold: **14 communities in both files**, none rare. `plan_name`↔`plan_code` is 1:1; product line matches prefix `SF`/`TH`/`DX`; each community maps to one metro and one permit authority.

No normalized category exists only in the scoring file. Scoring `home_id`s and the Aug–Dec 2025 start window are new, as expected.

**Mix shift (not a broken field):** scoring is more townhome (42% vs 29%), smaller (median sqft 1,680 vs 1,950), and only late-2025 starts. Spec rate is the same (~23%). Townhomes finish faster in history, so an unadjusted model can look pessimistic on the scoring set.


In [6]:
CATS = ["community", "metro", "permit_authority", "plan_name", "plan_code", "product_line",
        "garage_size", "basement_type", "lot_type", "site_manager", "data_source",
        "construction_status"]

def fold(s):
    return s.dropna().astype(str).str.strip().str.casefold()

prof = []
for col in CATS:
    for name, df in [("history", hist), ("predict", pred)]:
        s = df[col]
        raw = s.dropna().astype(str)
        prof.append({
            "column": col, "dataset": name,
            "unique_raw": int(s.nunique(dropna=False)),
            "unique_norm": int(fold(s).nunique()),
            "ws_rows": int((raw != raw.str.strip()).sum()),
            "min_n": int(raw.value_counts().min()) if len(raw) else 0,
        })
display(pd.DataFrame(prof).set_index(["column", "dataset"]))

print("community raw labels that collapse together (history):")
tmp = pd.DataFrame({"raw": hist.community, "fold": hist.community.str.strip().str.casefold()})
variants = tmp.drop_duplicates().groupby("fold")["raw"].apply(lambda x: sorted(x.unique()))
display(variants[variants.apply(len) > 1].to_frame("raw_variants"))

print("garage_size mixed encoding:")
display(pd.concat([
    hist.garage_size.value_counts().rename("history"),
    pred.garage_size.value_counts().rename("predict"),
], axis=1).fillna(0).astype(int))

only = []
for col in CATS:
    hset, pset = set(fold(hist[col])), set(fold(pred[col]))
    only.append({
        "column": col,
        "only_in_predict": sorted(pset - hset) or "—",
        "only_in_history": sorted(hset - pset) or "—",
    })
print("normalized labels only in one file (construction_status 'complete' is history-only, as expected):")
display(pd.DataFrame(only))

print("product / metro mix (%)")
mix = lambda df, col: (df[col].value_counts(normalize=True) * 100).round(1)
display(pd.DataFrame({"hist_product": mix(hist, "product_line"), "pred_product": mix(pred, "product_line")}).fillna(0))
display(pd.DataFrame({"hist_metro": mix(hist, "metro"), "pred_metro": mix(pred, "metro")}).fillna(0))
print("spec rate hist/pred:", round(hist.is_spec_home.mean(), 3), round(pred.is_spec_home.mean(), 3),
      "| median sqft hist/pred:", int(hist.sqft.median()), int(pred.sqft.median()))
print("predict start months:",
      pred_dt.construction_start_date.dt.to_period("M").value_counts().sort_index().to_dict())

print("sale_date missing vs spec (expected for specs; extra missings on non-specs):")
print("history"); display(pd.crosstab(hist.is_spec_home, hist.sale_date.isna(),
                                     rownames=["is_spec"], colnames=["sale_missing"]))
print("predict"); display(pd.crosstab(pred.is_spec_home, pred.sale_date.isna(),
                                     rownames=["is_spec"], colnames=["sale_missing"]))


unique_raw  unique_norm  ws_rows  min_n
column              dataset                                         
community           history          70           14      198      3
                    predict          45           14       25      1
metro               history           3            3        0   1396
                    predict           3            3        0    176
permit_authority    history           7            7        0    323
                    predict           7            7        0     35
plan_name           history          16           16        0    216
                    predict          16           16        0     20
plan_code           history          16           16        0    216
                    predict          16           16        0     20
product_line        history           3            3        0    727
                    predict           3            3        0     94
garage_size         history           6            6        0     75
                    predict           6            6        0     10
basement_type       history           3            2        0    978
                    predict           3            2        0    103
lot_type            history           3            3        0    603
                    predict           3            3        0     68
site_manager        history          18           18        0    210
                    predict          18           18        0     19
data_source         history           1            1        0   4684
                    predict           1            1        0    578
construction_status history           2            2        0    579
                    predict           1            1        0    578

community raw labels that collapse together (history):


,raw_variants
fold,
auburn meadows,"[ Auburn Meadows, AUBURN MEADOWS, Auburn Meadows, Auburn Meadows , auburn meadows]"
aurora highlands,"[ Aurora Highlands, AURORA HIGHLANDS, Aurora Highlands, Aurora Highlands , aurora high..."
barefoot lakes,"[ Barefoot Lakes, BAREFOOT LAKES, Barefoot Lakes, Barefoot Lakes , barefoot lakes]"
cranston ridge,"[ Cranston Ridge, CRANSTON RIDGE, Cranston Ridge, Cranston Ridge , cranston ridge]"
glenridding,"[ Glenridding, GLENRIDDING, Glenridding, Glenridding , glenridding]"
green valley,"[ Green Valley, GREEN VALLEY, Green Valley, Green Valley , green valley]"
keswick landing,"[ Keswick Landing, KESWICK LANDING, Keswick Landing, Keswick Landing , keswick landing]"
laurel green,"[ Laurel Green, LAUREL GREEN, Laurel Green, Laurel Green , laurel green]"
legacy gate,"[ Legacy Gate, LEGACY GATE, Legacy Gate, Legacy Gate , legacy gate]"


garage_size mixed encoding:


,history,predict
garage_size,,
2,1767,196
3,1363,119
1,1295,232
Double,99,10
Triple,85,11
Single,75,10


...

,hist_metro,pred_metro
metro,,
Calgary,35.30,37.50
Denver,34.90,32.00
Edmonton,29.80,30.40


spec rate hist/pred: 0.231 0.232 | median sqft hist/pred: 1950 1680
predict start months: {Period('2025-08', 'M'): 107, Period('2025-09', 'M'): 127, Period('2025-10', 'M'): 129, Period('2025-11', 'M'): 114, Period('2025-12', 'M'): 101}
sale_date missing vs spec (expected for specs; extra missings on non-specs):
history


sale_missing,False,True
is_spec,,
False,3099,504
True,0,1081


predict


sale_missing,False,True
is_spec,,
False,382,62
True,0,134


## 6. Field classification

**Availability at start** is separate from **whether the field should go into the model**.

| Availability | Meaning |
|---|---|
| **Safe** | Reasonably known on the start date; not a function of how long the job later takes |
| **Unsafe** | Unavailable until after start, or would leak the target / future progress |
| **Ambiguous** | Present in both files, but start-time availability cannot be confirmed from the data alone |

| `use_in_model` | Meaning |
|---|---|
| **yes** | Give this field (after later cleaning) to the model |
| **no** | Do not give this field to the model |
| **maybe** | Hold until a cleaning / leakage question is resolved |

A Safe field can still be `no` (id, constant) or `maybe` (redundant or sparse). An Unsafe field is always `no` as a feature. `construction_end_date` is used only to compute the target.


In [7]:
clf = pd.DataFrame([
    ["home_id", "Safe", "Assigned before/at start. Join key, not a duration leak.",
     "no", "Identifier only; no transferable signal."],
    ["community", "Safe", "Lot geography known at selection. Labels need normalizing.",
     "yes", ""],
    ["metro", "Safe", "Determined by community.",
     "yes", ""],
    ["permit_authority", "Safe", "Jurisdiction known before permitting; issue always precedes start.",
     "maybe", "1:1 with community after normalization; redundant if community is used."],
    ["lot_number", "Safe", "Known before start. Not unique across communities.",
     "no", "Local lot id; not unique and not transferable."],
    ["plan_name", "Safe", "Selected before construction.",
     "yes", ""],
    ["plan_code", "Safe", "1:1 with plan_name.",
     "maybe", "Duplicate of plan_name; keep one, not both."],
    ["product_line", "Safe", "Plan family (SF / TH / DX).",
     "yes", ""],
    ["beds", "Safe", "Plan spec. ~3% missing in both files; not status-dependent.",
     "yes", ""],
    ["baths", "Safe", "Plan spec; complete.",
     "yes", ""],
    ["half_baths", "Safe", "Plan spec; 0 = no powder room.",
     "yes", ""],
    ["sqft", "Safe", "Known at start. Extra-zero outliers are entry errors, not post-start revisions.",
     "yes", ""],
    ["stories", "Safe", "Plan spec (1 or 2).",
     "yes", ""],
    ["garage_size", "Safe", "Plan spec; encoding is mixed but pre-start.",
     "yes", ""],
    ["basement_type", "Safe", "Plan/lot spec. Literal 'None' = no basement, not unknown.",
     "yes", ""],
    ["lot_type", "Safe", "Flat / Corner / Walkout known before start.",
     "yes", ""],
    ["lot_size_sqft", "Safe", "Lot attribute. ~2% missing in both files.",
     "yes", ""],
    ["is_spec_home", "Safe", "Inventory vs contracted; populated on scoring homes.",
     "yes", ""],
    ["selected_options_value", "Ambiguous",
     "Populated on scoring homes, so a value exists at extract. Cannot tell contracted-at-start vs later upgrades.",
     "maybe", "May include mid-build upgrades; confirm it is the contracted-at-start amount."],
    ["site_manager", "Ambiguous",
     "Always populated, including scoring. Cannot prove the name is not rewritten mid-job.",
     "maybe", "May be reassigned after start; also likely collinear with community."],
    ["sale_date", "Safe",
     "Every recorded sale is on or before construction start, so a known sale is known at start. Missingness (all specs; extra non-specs) is a cleaning issue, not a leak.",
     "maybe", "Missing for all specs and 504/62 non-specs; needs a missingness rule before it is model-ready."],
    ["permit_application_date", "Safe", "Always present; always on/before issue and start.",
     "maybe", "Known at start, but a lag to issue/start is the useful form, not the raw calendar stamp."],
    ["permit_issue_date", "Safe", "Always present; every start is on/after issue.",
     "maybe", "Known at start, but a lag to start is the useful form, not the raw calendar stamp."],
    ["construction_start_date", "Safe",
     "As-of date of the prediction. Needed to define the problem; not itself a duration.",
     "maybe", "Useful for season, but the raw timestamp can overfit 2021-2025; scoring is only late 2025."],
    ["construction_end_date", "Unsafe",
     "Target endpoint. Empty for every scoring home and every in-progress history row.",
     "no", "Used only to compute the target (end - start), never as a feature."],
    ["final_inspection_date", "Unsafe",
     "Occurs at completion (typically 1-4 days before recorded end). Empty in scoring.",
     "no", "Post-start. Not the target. Empty in the scoring file."],
    ["change_order_count", "Unsafe",
     "Accumulates during build (in-progress mean 0.60 vs complete 1.82). 100% missing in scoring.",
     "no", "Post-start running count; not available for scoring homes."],
    ["trade_invoice_total", "Unsafe",
     "Running trade cost; in-progress totals much lower. 100% missing in scoring.",
     "no", "Post-start running total; not available for scoring homes."],
    ["data_source", "Safe", "Constant BUILDPRO_EXPORT. No information, no leak.",
     "no", "Constant; zero variance."],
    ["construction_status", "Unsafe",
     "Whether the job has finished -- a function of elapsed time and the snapshot. Scoring is 100% In Progress.",
     "no", "Filter for who has a target, not a feature."],
], columns=["field", "class", "availability_reasoning", "use_in_model", "model_note"])
display(clf)
print("availability:"); print(clf["class"].value_counts().to_string())
print("use_in_model:"); print(clf["use_in_model"].value_counts().to_string())


,field,class,availability_reasoning,use_in_model,model_note
0,home_id,Safe,"Assigned before/at start. Join key, not a duration leak.",no,Identifier only; no transferable signal.
1,community,Safe,Lot geography known at selection. Labels need normalizing.,yes,
2,metro,Safe,Determined by community.,yes,
3,permit_authority,Safe,Jurisdiction known before permitting; issue always precedes start.,maybe,1:1 with community after normalization; redundant if community is used.
4,lot_number,Safe,Known before start. Not unique across communities.,no,Local lot id; not unique and not transferable.
5,plan_name,Safe,Selected before construction.,yes,
6,plan_code,Safe,1:1 with plan_name.,maybe,"Duplicate of plan_name; keep one, not both."
7,product_line,Safe,Plan family (SF / TH / DX).,yes,
8,beds,Safe,Plan spec. ~3% missing in both files; not status-dependent.,yes,
9,baths,Safe,Plan spec; complete.,yes,


availability:
class
Safe         23
Unsafe        5
Ambiguous     2
use_in_model:
use_in_model
yes      14
no        8
maybe     8


## 7. Proposed cleaning-decision table

Proposals only. Nothing below has been applied to the raw files.


In [8]:
decisions = pd.DataFrame([
    ["Exact duplicate rows",
     "build_history: 25 extra rows / 25 home_ids",
     "duplicated()==25; each copied home_id is identical on every column",
     "Keep one row per home_id",
     "Export copies, not two builds; leaving them double-weights 25 homes",
     "nunique check supports full copies"],
    ["basement_type token 'None' read as missing",
     "basement_type: 1,654 hist (35.3%), 260 pred (45.0%)",
     "Literal read is None / Unfinished / Finished with no blanks; default read converts None→NaN",
     "Preserve 'None' (or recode to 'No basement') on load. Do not impute",
     "Valid third category, not MCAR missing",
     "Assumes no true unknowns were also stored as the word None"],
    ["garage_size mixed words and digits",
     "garage_size: hist Double 99 / Triple 85 / Single 75; pred 10 / 11 / 10 plus 1/2/3",
     "Words align with product line the same way as 1/2/3 (Single only on townhomes)",
     "Map Single→1, Double→2, Triple→3",
     "Same attribute, two encodings",
     "Assumes Double always means 2-car"],
    ["community case and whitespace",
     "community: 70 raw hist labels → 14; 45 raw pred → 14. Whitespace on 198 / 25 rows",
     "Each fold group is Title / UPPER / lower / padded variants of one name",
     "Strip + one canonical Title Case. Do not drop rows",
     "Otherwise WILDGRASS looks like an unseen category at score time",
     "No true aliases (e.g. River Bend) exist beyond case/space"],
    ["sqft extra-zero outliers",
     "sqft: 13 hist + 3 pred rows >10,000 (max 34,900 / 27,700)",
     "Each is ~10× plan median; IQR fence ~4,560 flags the same rows. home_ids in §3",
     "Prefer ÷10 when the result falls in that plan’s typical range; else manual review. Apply to both files",
     "Shared data-entry pattern, not a different product",
     "÷10 could be wrong if a row is truly multi-unit — spot-check a few ids"],
    ["End date before start",
     "7 Complete rows: H100392, H101045, H101204, H101439, H103515, H103689, H103849",
     "end-start in {-8 to -2}. Same 7 are the only inspection-after-end cases (inspection-start 108-313 days) -- evidence the end dates are wrong, not a second target",
     "Exclude from training until construction_end_date is corrected in source. Do not compute cycle time from inspection",
     "Target is always end - start; a negative value is unusable",
     "Dropping 7 is low bias. Do not silently swap in inspection as the target"],
    ["In-progress history has no target",
     "579 hist rows (12.4%), all missing end and inspection",
     "Status × end-missing is diagonal. Starts 2024-08-04 to 2025-07-31 (32 from 2024, 547 from 2025)",
     "Exclude from a fully observed target. Do not impute an end date",
     "Right-censored at the snapshot, not missing at random",
     "Excluding them length-biases recent cohorts (see next)"],
    ["Snapshot truncation / short 2025 completes",
     "All files. Hist dates stop 2025-07-31; pred starts 2025-08-01. 156 of 703 year-2025 hist starts completed",
     "Those 156 have mean cycle ~130d vs ~187d overall",
     "Document the cut. Later, consider excluding very recent completes or a delayed-entry rule",
     "A May 2025 start can only appear Complete if it finished by 31 Jul 2025",
     "Cut date is inferred from the files, not documented in them"],
    ["Non-spec homes missing sale_date",
     "sale_date: all 1,081 / 134 specs missing, plus 504 hist / 62 pred non-specs",
     "Crosstab is_spec × sale missing; spec with a sale_date count is 0",
     "Treat spec missing as expected. Do not impute non-spec missings until the meaning is confirmed",
     "Two missingness mechanisms; collapsing them mixes inventory with broken sold records",
     "Some non-spec missings might be mis-flagged specs"],
    ["Missing beds (~3%)",
     "beds: 141 hist (3.01%), 17 pred (2.94%), spread across plans; 125 of 141 hist missings are Complete",
     "Missing rate matches across files; values otherwise 2–5 integers",
     "Later: impute from plan_code, not a global mean. Not applied now",
     "Looks like a plan attribute with data-entry gaps, not a leak pattern",
     "Some single-family plans span 3–5 beds, so plan-only impute is coarse"],
    ["Missing lot_size_sqft (~2%)",
     "lot_size_sqft: 97 hist (2.07%), 12 pred (2.08%), spread across communities",
     "2 overlap with sqft outliers; no zeros or negatives",
     "Later: impute from community × lot_type, or keep with an indicator. Not applied now",
     "Same rate in scoring, so it must be handled; unlikely informative missingness for duration",
     "Walkout/corner lots differ — a global median would blur that"],
    ["Post-start fields in history, empty in predict",
     "change_order_count, trade_invoice_total, construction_end_date, final_inspection_date, construction_status",
     "Four fields 100% missing in predict; status 100% In Progress. In-progress invoices/COs systematically lower",
     "Do not use as features. construction_end_date is the target only. Status is a filter. Inspection is not the target",
     "Unavailable at start on scoring homes; completed totals leak duration",
     "A later *planned* budget field would be different and could be Safe"],
    ["selected_options_value availability",
     "selected_options_value: 0 missing; spec mean ~8.4k vs sold ~33k; 74 hist rows above IQR fence, max 145,000",
     "Populated on scoring homes. High tail includes townhomes",
     "Candidate feature only after ops confirms it is contracted-at-start. Review $100k+ separately",
     "Could be Safe and useful, or a mild leak if it includes mid-build upgrades",
     "High townhome option totals may be mis-keyed"],
    ["site_manager as a feature",
     "site_manager: 18 people, min hist count 210, none new in predict",
     "Fully populated in both files",
     "Eligible later if ops confirms the manager of record is assigned by start",
     "Crew skill could affect duration; reassignment would make the stored name post-start",
     "May be collinear with community"],
    ["Target is end - start only",
     "4,105 Complete rows; 7 of them have negative cycle_days",
     "Target formula is fixed: construction_end_date - construction_start_date. Inspection is adjacent (median 3 days earlier) but is not the target",
     "Compute cycle_days only from end - start. Handle the 7 negative rows by exclusion or source correction, not by switching formula",
     "Matches the required construction cycle-time definition",
     "The 7 inverted ends have no valid target until construction_end_date is fixed"],
    ["Train vs score mix shift",
     "All rows. Predict 41.9% townhome vs 29.2% hist; all starts Aug–Dec 2025",
     "Median sqft 1,950 vs 1,680; 2022 hist cycles were the longest",
     "No row dropping. Later: stratify evaluation by product_line; use time-aware validation",
     "Population shift, not a broken field",
     "Random 2021–2025 splits will not mimic Aug–Dec 2025 scoring"],
    ["lot_number is not a unique key",
     "lot_number reuses 1–419; 1,474 community+lot collisions in history",
     "home_id unique after de-duplication except the 25 exact copies",
     "Use home_id as the only key",
     "Lot numbers are local to a community",
     "None if home_id is used"],
    ["Dtype mismatch on all-null predict columns",
     "end/inspection dates object vs float; change_order_count int vs float",
     "Those predict columns have 0 non-null values",
     "Parse dates explicitly later; exclude these columns as features",
     "Prevents silent concat bugs",
     "None if parsers are explicit"],
], columns=["issue", "affected", "evidence", "proposed_treatment", "reasoning", "risks_assumptions"])

pd.set_option("display.max_colwidth", 140)
display(decisions)


,issue,affected,evidence,proposed_treatment,reasoning,risks_assumptions
0,Exact duplicate rows,build_history: 25 extra rows / 25 home_ids,duplicated()==25; each copied home_id is identical on every column,Keep one row per home_id,"Export copies, not two builds; leaving them double-weights 25 homes",nunique check supports full copies
1,basement_type token 'None' read as missing,"basement_type: 1,654 hist (35.3%), 260 pred (45.0%)",Literal read is None / Unfinished / Finished with no blanks; default read converts None→NaN,Preserve 'None' (or recode to 'No basement') on load. Do not impute,"Valid third category, not MCAR missing",Assumes no true unknowns were also stored as the word None
2,garage_size mixed words and digits,garage_size: hist Double 99 / Triple 85 / Single 75; pred 10 / 11 / 10 plus 1/2/3,Words align with product line the same way as 1/2/3 (Single only on townhomes),"Map Single→1, Double→2, Triple→3","Same attribute, two encodings",Assumes Double always means 2-car
3,community case and whitespace,community: 70 raw hist labels → 14; 45 raw pred → 14. Whitespace on 198 / 25 rows,Each fold group is Title / UPPER / lower / padded variants of one name,Strip + one canonical Title Case. Do not drop rows,Otherwise WILDGRASS looks like an unseen category at score time,No true aliases (e.g. River Bend) exist beyond case/space
4,sqft extra-zero outliers,"sqft: 13 hist + 3 pred rows >10,000 (max 34,900 / 27,700)","Each is ~10× plan median; IQR fence ~4,560 flags the same rows. home_ids in §3",Prefer ÷10 when the result falls in that plan’s typical range; else manual review. Apply to both files,"Shared data-entry pattern, not a different product",÷10 could be wrong if a row is truly multi-unit — spot-check a few ids
5,End date before start,"7 Complete rows: H100392, H101045, H101204, H101439, H103515, H103689, H103849",end-start in {-8 to -2}. Same 7 are the only inspection-after-end cases (inspection-start 108-313 days) -- evidence the end dates are wr...,Exclude from training until construction_end_date is corrected in source. Do not compute cycle time from inspection,Target is always end - start; a negative value is unusable,Dropping 7 is low bias. Do not silently swap in inspection as the target
6,In-progress history has no target,"579 hist rows (12.4%), all missing end and inspection","Status × end-missing is diagonal. Starts 2024-08-04 to 2025-07-31 (32 from 2024, 547 from 2025)",Exclude from a fully observed target. Do not impute an end date,"Right-censored at the snapshot, not missing at random",Excluding them length-biases recent cohorts (see next)
7,Snapshot truncation / short 2025 completes,All files. Hist dates stop 2025-07-31; pred starts 2025-08-01. 156 of 703 year-2025 hist starts completed,Those 156 have mean cycle ~130d vs ~187d overall,"Document the cut. Later, consider excluding very recent completes or a delayed-entry rule",A May 2025 start can only appear Complete if it finished by 31 Jul 2025,"Cut date is inferred from the files, not documented in them"
8,Non-spec homes missing sale_date,"sale_date: all 1,081 / 134 specs missing, plus 504 hist / 62 pred non-specs",Crosstab is_spec × sale missing; spec with a sale_date count is 0,Treat spec missing as expected. Do not impute non-spec missings until the meaning is confirmed,Two missingness mechanisms; collapsing them mixes inventory with broken sold records,Some non-spec missings might be mis-flagged specs
9,Missing beds (~3%),"beds: 141 hist (3.01%), 17 pred (2.94%), spread across plans; 125 of 141 hist missings are Complete",Missing rate matches across files; values otherwise 2–5 integers,"Later: impute from plan_code, not a global mean. Not applied now","Looks like a plan attribute with data-entry gaps, not a leak pattern","Some single-family plans span 3–5 beds, so plan-only impute is coarse"


## 8. Findings to confirm before cleaning

1. **One extract, two files.** 4,684 history x 30 cols; 578 scoring x same cols. Cut date appears to be 2025-07-31 / 2025-08-01.
2. **Target** is always `construction_end_date - construction_start_date`. Computable for 4,105 completes (median 178 days), not for 579 in-progress or any scoring home. Seven completes have impossible negative cycles and cannot be used until their end dates are corrected.
3. **Do not give the model:** end date (target only), inspection date, change-order count, trade invoice total, construction status, home_id, lot_number, data_source.
4. **Shared dirt, no new scoring categories:** community case/space (70 to 14), garage words vs digits, basement `"None"` vs pandas `NaN`, 16 extra-zero sqft values (3 in scoring), 25 exact duplicate history rows.
5. **Scoring mix is different:** more townhomes, smaller homes, only late-2025 starts.

**Needs a decision before any cleaning starts:** what to do with the 7 inverted ends (drop vs source-correct; do not switch the target formula); in-progress rows exclude-only vs later censoring; whether 2025 completes are allowed into training; confirm `"None"` = no basement; garage word-to-number map; whether options value and site manager are start-of-job; meaning of non-spec missing sale dates; sqft extra zeros (divide-by-10 vs review vs drop); de-duplicate the 25 copied rows.

Raw files are unchanged.
